# [데이터 통합] 이벤트 로그 & 교정 기록 병합

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

해결 하고자 하는 문제 = 재방문율
방향성을 가지고 데이터 탐색


In [ ]:
#필요없는 컬럼
drop_cols_df2 = [
    'browser_version', 'email', 'geo_source', 'name',
    'initial_utm_campaign', 'initial_utm_content', 'initial_utm_id',
    'initial_utm_medium', 'initial_utm_source', 'initial_utm_term',
    'initial_utm_source_platform'
]

drop_cols_df3 = [
    'browser_is_whale', 'browser_is_chrome', 'browser_is_edge',
    'email', 'name', 'browser_version', 'insert_id',
    'screen_height', 'screen_width', 'search_engine', 'lib_version',
    'mp_api_endpoint', 'mp_lib', 'mp_loader', 'extension_version',
    'llm_provider', 'llm_version', 'position', 'platform', 'trigger'
]

1번 데이터셋 불러오기  

In [ ]:
df1 = pd.read_json('../data/raw/1. 사용자별 사용량 데이터.json')

df1 데이터프레임의 '_id' 컬럼이 MongoDB 스타일의 ObjectID 형식   
JSON 파일을 pandas로 읽어들일 때, 이런 중첩된 딕셔너리 구조가 그대로 로드 하는것을 방지   
df1의 '_id' 컬럼을 문자열로 변환 후, '_id_str' 컬럼에 저장  
'recent_execution_date'날짜 값을 추출한 후 datetime 형식으로 변환  

In [ ]:
df1['_id_str'] = df1['_id'].apply(lambda x: x.get('$oid') if isinstance(x, dict) else None)
df1['recent_execution_date'] = df1['recent_execution_date'].apply(lambda x: x.get('$date') if isinstance(x, dict) else None)
df1['recent_execution_date'] = pd.to_datetime(df1['recent_execution_date'])

2번 데이터셋 불러오기  

In [ ]:
df2 = pd.read_json('../data/raw/2. 사용자 클라이언트 데이터.json', lines=True) 

In [ ]:
df2['user_id'].value_counts()

user_id
66a1d5003d6795f27f0f18d1                1
581c9fb8-4582-4bac-a07b-5ff0fbecdb2f    1
cf6634ec-dc5d-4888-9db2-33f42d146075    1
6862433ec0ffcb60d473b76a                1
42ff562b-0fab-4a59-ae2a-99f306a5e4ba    1
26b29a63-9ebf-46a1-8422-d53080c63248    1
6780b8bdaefa1c680bd2ce48                1
24128ee4-fee6-48f0-942d-aefdb60f01fe    1
391be5a2-94ea-447c-8bdc-13cbbb28fc30    1
67a1bd41da85d0824a35b81e                1
9c416198-ef17-490e-87fd-f8966b48ce1a    1
40743a9e-3a77-41e0-9735-c192e56f6615    1
86a42fa9-e421-499b-9c03-c9cf181413ef    1
8c75d1e2-42dd-410e-ab0b-6121ba4bf06d    1
65f2a3f0e6f6fb51657f87cb                1
4268b5f7-4a00-4c69-8475-2b7c55dd5f46    1
667ba2752d067733072d806f                1
685e093dc0ffcb60d473b4eb                1
8465f7fe-14fc-49ac-9fef-448d95a96092    1
65efb59ea0fb3027ba86d551                1
4d96113c-b3d6-493f-aac1-ec0f176e877b    1
68217c9062314855e4c117e0                1
0f5e93ac-ae1d-4e71-bfa0-e2cd9e5a70c5    1
0cdfdb58-f901-4cca-94a9-7a

df2에 'properties' 컬럼을 중첩된 JSON으로 정규화  
'distinct_id'가 중복되면 삭제하고, 원본 df2와 병합해서 충돌 방지  
중첩된 속성을 평탄화  

In [ ]:
if 'properties' in df2.columns:
    properties_df = pd.json_normalize(df2['properties'])
    if 'distinct_id' in properties_df.columns:
        properties_df = properties_df.drop('distinct_id', axis=1)  
    df2 = pd.concat([df2.drop('properties', axis=1), properties_df], axis=1)

3번 데이터셋 불러오기

In [ ]:
df3 = pd.read_json('../data/raw/3. 이벤트 데이터.json', lines=True)

df3의 'user_id' 컬럼을 문자열로 변환 후 'user_id_str' 컬럼에 저장  
딕셔너리 형식에서 '$oid' 값을 추출

In [ ]:
df3['user_id_str'] = df3['user_id'].apply(lambda x: x.get('$oid') if isinstance(x, dict) else x)

df3에 'properties' 컬럼이 있으면 이를 정규화  
'distinct_id'와 'user_id'가 중복되면 삭제, 원본 df3와 병합  
df2와 유사한 평탄화  

In [ ]:
if 'properties' in df3.columns:
    props_df3 = pd.json_normalize(df3['properties'])
    for col in ['distinct_id', 'user_id']:
        if col in props_df3.columns:
            props_df3 = props_df3.drop(col, axis=1)
    df3 = pd.concat([df3.drop('properties', axis=1), props_df3], axis=1)

In [ ]:
df3

,event_name,distinct_id,insert_id,device_id,user_id,time,user_id_str,browser,browser_version,city,...,user_propmt_text,index,mp_keyword,app_version,platform,$distinct_id_before_identity,terminal_word,$had_persisted_distinct_id,utm_id,fbclid
0,pageview_ad inflow,$device:8a4b719c-aed1-4036-a7c9-7a9eddb218f1,i8gg3ep5wboqqbfn,8a4b719c-aed1-4036-a7c9-7a9eddb218f1,None,1759468140,None,Chrome,140,Gunpo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,editor_run_paraphrasing,68df0654e52d1b541719dcbc,5t77elnh10o7s2t3,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759449676,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,editor_run_paraphrasing,68df0654e52d1b541719dcbc,jko789ul6jqt62h4,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759449816,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,editor_run_paraphrasing,68df0654e52d1b541719dcbc,yizypn6xsu55kk3d,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759450020,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,editor_run_paraphrasing,68df0654e52d1b541719dcbc,cddsv7hlm4uxbev0,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759450144,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62092,run_paraphrasing,$device:417e55e4-84dd-433e-9375-c16a5dd3419a,tk3wqy1miuzp10i9,417e55e4-84dd-433e-9375-c16a5dd3419a,None,1750828427,None,Chrome,136,Seo-gu,...,NaN,NaN,NaN,NaN,NaN,NaN,none,NaN,NaN,NaN
62093,run_paraphrasing,$device:675a9117-17a7-41f7-b86c-5d8853f6d278,mnq25krnefxethie,675a9117-17a7-41f7-b86c-5d8853f6d278,None,1750861706,None,Chrome,137,Seo-gu,...,NaN,NaN,NaN,NaN,NaN,NaN,none,NaN,NaN,NaN
62094,run_paraphrasing,$device:e4bca22e-674b-47fe-9ae6-6142b8651458,ijqsofwk1lqpvkhd,e4bca22e-674b-47fe-9ae6-6142b8651458,None,1750815835,None,Chrome,137,Suwon,...,NaN,NaN,NaN,NaN,NaN,NaN,none,NaN,NaN,NaN
62095,selected_paraphrasing,$device:e4bca22e-674b-47fe-9ae6-6142b8651458,3k9xzitg6wmul725,e4bca22e-674b-47fe-9ae6-6142b8651458,None,1750815838,None,Chrome,137,Suwon,...,NaN,2.0,NaN,NaN,NaN,NaN,none,NaN,NaN,NaN


df1과 df2를 병합하여 user_merged 데이터프레임을 만듬  
'_id_str'과 'distinct_id'를 키로 사용하며, outer join으로 모든 레코드를 포함  
사용자 사용량과 클라이언트 데이터를 결합  

In [ ]:
user_merged = pd.merge(df1, df2, left_on='_id_str', right_on='distinct_id', how='outer')

In [ ]:
user_merged

,_id,count,recent_execution_date,_id_str,distinct_id,browser,browser_version,city,country_code,email,...,$role,initial_utm_campaign,initial_utm_content,initial_utm_id,initial_utm_medium,initial_utm_source,initial_utm_term,$device_id,identify,initial_utm_source_platform
0,NaN,NaN,NaT,NaN,$device:001b2cbe-4357-48c8-82dd-9011abaf6768,Chrome,140.0,Changwon,KR,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaT,NaN,$device:00224af8-e07c-46a8-ab45-d5e7534dbff8,Chrome,138.0,Jongno-gu,KR,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaT,NaN,$device:0022e7e0-bdf3-4f71-bbb1-53f1f2712c4a,Chrome,134.0,Yongin-si,KR,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaT,NaN,$device:0025c176-f779-46bd-bc05-c7ec48709419,Chrome,139.0,Changwon,KR,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaT,NaN,$device:00429333-eef4-4677-99a4-a37f098910aa,Samsung Internet,28.0,Yongin-si,KR,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6794,NaN,NaN,NaT,NaN,fc3225d0-0972-434e-b676-1636cf059d3c,Chrome,134.0,Nagoya,JP,None,...,NaN,UA_web_pc_kakao,NaN,NaN,NETWORK,kakao,NaN,NaN,NaN,NaN
6795,NaN,NaN,NaT,NaN,fc4f0c45-4f59-4782-bf3a-6af6c66c07c3,Chrome,137.0,Suwon,KR,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6796,NaN,NaN,NaT,NaN,fdbda59e-9163-48ff-8786-f686ab38d142,Chrome,137.0,Yuseong-gu,KR,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6797,NaN,NaN,NaT,NaN,ff6329da-9dfb-4e07-88ea-4f914eefc802,Chrome,135.0,Gangseo-gu,KR,None,...,NaN,NaN,NaN,NaN,NaN,chatgpt.com,NaN,NaN,NaN,NaN


df3과 user_merged에 병합 키('merge_key')를 생성(병합할때 안정성 확보)  
'user_id_str'을 사용  
left join으로 병합 후 full_merged를 만들고 불필요한 키 컬럼을 삭제  
이벤트 데이터와 사용자 데이터를 통합  

In [ ]:
df3['merge_key'] = df3['user_id_str'].fillna(df3['distinct_id'])
user_merged['merge_key'] = user_merged['_id_str'].fillna(user_merged['distinct_id'])
full_merged = pd.merge(df3, user_merged, on='merge_key', how='left', suffixes=('_event', '_user'))
full_merged.drop(['merge_key_event', 'merge_key_user', 'merge_key'], axis=1, errors='ignore', inplace=True)

In [ ]:
full_merged

,event_name,distinct_id_event,insert_id,device_id,user_id_event,time,user_id_str,browser_event,browser_version_event,city_event,...,$role,initial_utm_campaign,initial_utm_content,initial_utm_id,initial_utm_medium,initial_utm_source,initial_utm_term,$device_id,identify,initial_utm_source_platform
0,pageview_ad inflow,$device:8a4b719c-aed1-4036-a7c9-7a9eddb218f1,i8gg3ep5wboqqbfn,8a4b719c-aed1-4036-a7c9-7a9eddb218f1,None,1759468140,None,Chrome,140,Gunpo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,editor_run_paraphrasing,68df0654e52d1b541719dcbc,5t77elnh10o7s2t3,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759449676,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,...,1.0,NaN,NaN,NaN,naver_news,2753087,NaN,NaN,NaN,NaN
2,editor_run_paraphrasing,68df0654e52d1b541719dcbc,jko789ul6jqt62h4,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759449816,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,...,1.0,NaN,NaN,NaN,naver_news,2753087,NaN,NaN,NaN,NaN
3,editor_run_paraphrasing,68df0654e52d1b541719dcbc,yizypn6xsu55kk3d,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759450020,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,...,1.0,NaN,NaN,NaN,naver_news,2753087,NaN,NaN,NaN,NaN
4,editor_run_paraphrasing,68df0654e52d1b541719dcbc,cddsv7hlm4uxbev0,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759450144,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,...,1.0,NaN,NaN,NaN,naver_news,2753087,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62092,run_paraphrasing,$device:417e55e4-84dd-433e-9375-c16a5dd3419a,tk3wqy1miuzp10i9,417e55e4-84dd-433e-9375-c16a5dd3419a,None,1750828427,None,Chrome,136,Seo-gu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62093,run_paraphrasing,$device:675a9117-17a7-41f7-b86c-5d8853f6d278,mnq25krnefxethie,675a9117-17a7-41f7-b86c-5d8853f6d278,None,1750861706,None,Chrome,137,Seo-gu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62094,run_paraphrasing,$device:e4bca22e-674b-47fe-9ae6-6142b8651458,ijqsofwk1lqpvkhd,e4bca22e-674b-47fe-9ae6-6142b8651458,None,1750815835,None,Chrome,137,Suwon,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62095,selected_paraphrasing,$device:e4bca22e-674b-47fe-9ae6-6142b8651458,3k9xzitg6wmul725,e4bca22e-674b-47fe-9ae6-6142b8651458,None,1750815838,None,Chrome,137,Suwon,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


중복된 컬럼 이름을 검사, 중복 시 숫자 접미사를 추가해서 고유화  
해시 가능한 컬럼를 선택, 처음 10개 컬럼으로 중복 행을 제거  
데이터 중복을 정리

In [ ]:
from collections import Counter
col_count = Counter(full_merged.columns)
if any(count > 1 for count in col_count.values()):
    new_cols = []
    seen = {}
    for col in full_merged.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    full_merged.columns = new_cols

hashable_cols = [col for col in full_merged.columns 
                 if full_merged.dtypes[col] in ['int64', 'float64', 'object', 'datetime64[ns]'] 
                 and not full_merged[col].apply(lambda x: isinstance(x, (dict, list))).any()]

full_merged = full_merged.drop_duplicates(subset=hashable_cols[:10])

데이터 클리닝  
'time' 컬럼을 한국 시간으로 변환  
NaN 값을 채우며(지리 정보는 'Unknown', 응답 시간은 중위수, 입력 길이는 0)  
불필요한 컬럼을 삭제  
익명 사용자 플래그('is_anon') 추가, 'count' 컬럼을 정수형으로 변환  
이상치 클리핑

In [ ]:
if 'time' in full_merged.columns:
    full_merged['time_dt'] = pd.to_datetime(full_merged['time'], unit='s', utc=True).dt.tz_convert('Asia/Seoul')

if 'recent_execution_date' in full_merged.columns:
    full_merged['recent_execution_date'] = full_merged['recent_execution_date'].apply(lambda x: x.get('$date') if isinstance(x, dict) else None)
    full_merged['recent_execution_date'] = pd.to_datetime(full_merged['recent_execution_date'])

if 'mp_api_timestamp_ms' in full_merged.columns:
    full_merged.drop('mp_api_timestamp_ms', axis=1, inplace=True)

geo_cols = ['city', 'region', 'country_code', 'timezone']
for col in geo_cols:
    if col in full_merged.columns:
        full_merged[col] = full_merged[col].fillna('Unknown')

if 'response_time_ms' in full_merged.columns:
    median_response = full_merged['response_time_ms'].median()
    full_merged['response_time_ms'] = full_merged['response_time_ms'].fillna(median_response)

if 'input_sentence_length' in full_merged.columns:
    full_merged['input_sentence_length'] = full_merged['input_sentence_length'].fillna(0)

if 'browser' in full_merged.columns:
    full_merged['browser'] = full_merged['browser'].fillna('Unknown')

drop_cols = ['mp_lib', 'mp_api_endpoint', 'mp_loader', 'lib_version', 'mp_processing_time_ms', 'insert_id', 'device_id_event', '_id', '_id_str']
full_merged = full_merged.drop(columns=[col for col in drop_cols if col in full_merged.columns], errors='ignore')

if 'distinct_id' in full_merged.columns:
    full_merged['is_anon'] = full_merged['distinct_id'].str.startswith('$device:')

if 'count' in full_merged.columns:
    full_merged['count'] = full_merged['count'].astype('Int64')

if 'response_time_seconds' in full_merged.columns:
    full_merged['response_time_seconds'] = pd.to_numeric(full_merged['response_time_seconds'], errors='coerce')
    
if 'count' in full_merged.columns:
    percentile_99 = full_merged['count'].quantile(0.99)
    full_merged['count'] = full_merged['count'].clip(upper=percentile_99)

print("Shape after cleaning:", full_merged.shape)
print(full_merged.info())
full_merged.head()

Shape after cleaning: (62097, 107)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62097 entries, 0 to 62096
Columns: 107 entries, event_name to time_dt
dtypes: Int64(1), datetime64[ns, Asia/Seoul](1), datetime64[ns](1), float64(8), int64(3), object(93)
memory usage: 50.8+ MB
None


,event_name,distinct_id_event,device_id,user_id_event,time,user_id_str,browser_event,browser_version_event,city_event,current_url,...,initial_utm_campaign,initial_utm_content,initial_utm_id,initial_utm_medium,initial_utm_source,initial_utm_term,$device_id,identify,initial_utm_source_platform,time_dt
0,pageview_ad inflow,$device:8a4b719c-aed1-4036-a7c9-7a9eddb218f1,8a4b719c-aed1-4036-a7c9-7a9eddb218f1,None,1759468140,None,Chrome,140,Gunpo,https://www.sentencify.ai/ko?gad_source=1&gad_...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-10-03 14:09:00+09:00
1,editor_run_paraphrasing,68df0654e52d1b541719dcbc,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759449676,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,https://www.sentencify.ai/ko/editor/write,...,NaN,NaN,NaN,naver_news,2753087,NaN,NaN,NaN,NaN,2025-10-03 09:01:16+09:00
2,editor_run_paraphrasing,68df0654e52d1b541719dcbc,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759449816,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,https://www.sentencify.ai/ko/editor/write,...,NaN,NaN,NaN,naver_news,2753087,NaN,NaN,NaN,NaN,2025-10-03 09:03:36+09:00
3,editor_run_paraphrasing,68df0654e52d1b541719dcbc,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759450020,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,https://www.sentencify.ai/ko/editor/write,...,NaN,NaN,NaN,naver_news,2753087,NaN,NaN,NaN,NaN,2025-10-03 09:07:00+09:00
4,editor_run_paraphrasing,68df0654e52d1b541719dcbc,ab4c83ae-2c34-4ab7-8f02-a07eb5d85b8a,68df0654e52d1b541719dcbc,1759450144,68df0654e52d1b541719dcbc,Whale Browser,4.33.325.17,Goyang-si,https://www.sentencify.ai/ko/editor/write,...,NaN,NaN,NaN,naver_news,2753087,NaN,NaN,NaN,NaN,2025-10-03 09:09:04+09:00


다 같이 필요 없다고 정의한 drop_cols_df2와 drop_cols_df3를 합쳐 중복 제거한 리스트를 만들고, full_merged에서 해당 컬럼들을 삭제

In [ ]:
drop_cols = list(set(drop_cols_df2 + drop_cols_df3))

full_merged = full_merged.drop(columns=[col for col in drop_cols if col in full_merged.columns])

In [ ]:
full_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62097 entries, 0 to 62096
Data columns (total 87 columns):
 #   Column                             Non-Null Count  Dtype                     
---  ------                             --------------  -----                     
 0   event_name                         62097 non-null  object                    
 1   distinct_id_event                  62097 non-null  object                    
 2   device_id                          62097 non-null  object                    
 3   user_id_event                      45991 non-null  object                    
 4   time                               62097 non-null  int64                     
 5   user_id_str                        45991 non-null  object                    
 6   browser_event                      62097 non-null  object                    
 7   browser_version_event              62096 non-null  object                    
 8   city_event                         60807 non-null  objec

병합된 데이터프레임을 'full_merged.csv' 파일로 저장

In [ ]:
full_merged.to_csv('full_merged.csv', index=False, encoding='utf-8-sig')

In [ ]:
print("Usage Statistics:")
print(user_merged['count'].describe())

Usage Statistics:
count      399.000000
mean       135.285714
std        874.164131
min          1.000000
25%          2.000000
50%          7.000000
75%         28.500000
max      11595.000000
Name: count, dtype: float64
